# Nurse Rostering 

## I. Khái niệm

- Bài toán sắp xếp điều dưỡng - $NRP$ có mục tiêu tạo ra một lịch làm việc khả thi cho $n$ điều dưỡng trong một giai đoạn kế hoạch gồm $d$ ngày bằng `SAT solver`

- $X_{i,j,k}$ là nhân viên $i$ trong ngày $j$ làm việc ca k (hoặc không làm ca nào)
    - $1 \le i \le n$: Vì có $n$ nhân viên
    - $1 \le j \le d$: Vì có $d$ ngày
    - $k \in \{D, E, N, O\}$ : $D$ là ca ngày, $E$ là ca tối, $N$ là ca đêm, $O$ là không làm ca nào trong ngày

## II. Cách chuyển ALK thành AMK

1. Ý tưởng:
    - Ít nhất $k$ tương đương với nhiều nhất $w - k$, "Tối đa $k$ biến trong số $w$ biến mang giá trị $True$" tương đương với ràng buộc "Ít nhất $w−k$ biến trong số $w$ biến đó mang giá trị $False$"

$$ \bigwedge_{i=0}^{n-w}\left(\sum_{j=i+1}^{i+w} X_j \ge k\right) \equiv \bigwedge_{i=0}^{n-w}\left(\sum_{j=i+1}^{i+w} \neg X_j \le w - k\right)\qquad 1 \le k < w$$

## III.Các ràng buộc

1. **Mỗi ngày** mỗi điều dưỡng làm **tối đa 1 ca làm việc**
    - Dạng triển khai:
        - Ngày 1 người 1 làm tối đa 1 ca: $X_{1,1,D} + X_{1,1,E} + X_{1,1,N} + X_{1,1,O} = 1$ \
          $...$
    - Dạng tổng quát:
      $$ \bigwedge_{i = 1} ^ n \bigwedge_{j = 1}^d \sum_{k \in \{D, E, N, O\}} X_{i, j, k} = 1$$
> thực tế nếu không có điều kiện (1) phải có điều kiện nếu $X_{i,j,O} = 1 \leftrightarrow X_{i,j,D} + X_{i, j, E} + X_{i, j, N} = 0$. Tuy nhiên điều kiện (1) đã bao gồm điều kiện trên. Và nếu bỏ điều kiện (1) này bài toán sẽ **phức tạp hơn nhiều**

2. Trong mỗi **7 ngày liên tiếp**, mỗi nhân viên làm **tối đa 6 ca làm việc**
   - Tương đương: **phủ định** của **ngày nghỉ** là **ngày làm việc**  \
     $$\bigwedge_{i = 1}^n (\bigwedge_{j = 1}^ {d - 6} \sum_{j'= j} ^ {j + 6} \neg X_{i, j', O} \le 6 )$$
    - Phần trong dấu ngoặc chính là các dàng buộc dạng `cầu thang` (ladder), phép $\land$ ở ngoài có là bởi vì có n nhân viên khác nhau

3. Trong **14 ngày liên tiếp**, phải có **ít nhất 4 ngày nghỉ**
$$\bigwedge_{i = 1}^n (\bigwedge_{j = 1}^ {d - 13} \sum_{j'= j} ^ {j + 13}  X_{i, j', O} \ge 4 ) \equiv \bigwedge_{i = 1}^n (\bigwedge_{j = 1}^ {d - 13} \sum_{j'= j} ^ {j + 13}  \neg X_{i, j', O} \le 10) $$

4. Trong **14 ngày liên tiếp**, phải có **ít nhất 4 ca tối**
   $$\bigwedge_{i = 1}^n (\bigwedge_{j = 1}^ {d - 13} \sum_{j'= j} ^ {j + 13}  X_{i, j', N} \ge 4 ) \equiv \bigwedge_{i = 1}^n (\bigwedge_{j = 1}^ {d - 13} \sum_{j'= j} ^ {j + 13} \neg X_{i, j', N} \le 10)$$

5. Trong **14 ngày liên tiếp**, có **tối đa 8 ca tối**
     $$\bigwedge_{i = 1}^n (\bigwedge_{j = 1}^ {d - 13} \sum_{j'= j} ^ {j + 13}  X_{i, j', E} \le 8 )$$

6. Trong mỗi **28 ngày liên tiếp**, phải có **ít nhất 20 ca làm việc**
   - Tương đương: **phủ định** của **ngày nghỉ** là **ngày làm việc**  \
     $$\bigwedge_{i = 1}^n (\bigwedge_{j = 1}^ {d - 27} \sum_{j'= j} ^ {j + 27}  \neg X_{i, j', O} \ge 20 ) \equiv \bigwedge_{i = 1}^n (\bigwedge_{j = 1}^ {d - 27} \sum_{j'= j} ^ {j + 27}   X_{i, j', O} \le 8 )$$

7. Trong mỗi **14 ngày liên tiếp**, có **tối đa 4 ca đêm**
   $$\bigwedge_{i = 1}^n (\bigwedge_{j = 1}^ {d - 13} \sum_{j'= j} ^ {j + 13}  X_{i, j', N} \le 4 )$$

8. Trong mỗi **14 ngày liên tiếp**, phải có **ít nhất 1 ca đêm**
   $$\bigwedge_{i = 1}^n (\bigwedge_{j = 1}^ {d - 13} \sum_{j'= j} ^ {j + 13}  X_{i, j', N} \ge 1 ) \equiv \bigwedge_{i = 1}^n (\bigwedge_{j = 1}^ {d - 13} \sum_{j'= j} ^ {j + 13}  \neg X_{i, j', N} \le 13 )$$

9. Trong **7 ngày liên tiếp**, phải có **ít nhất 2 ca tối hoặc ca đêm**
   $$\bigwedge_{i = 1}^n (\bigwedge_{j = 1}^ {d - 6} \sum_{j'= j} ^ {j + 6}  X_{i, j', E} + X_{i, j', N} \ge 2 ) \equiv  \bigwedge_{i = 1}^n (\bigwedge_{m = 1}^ {2d - 11} \sum_{j'= m} ^ {j + 11}  Y_{i, z} \ge 2 )  \equiv  \bigwedge_{i = 1}^n (\bigwedge_{m = 1}^ {2d - 11} \sum_{j'= m} ^ {j + 11}  \neg Y_{i, z} \le 5 )$$
   Trong đó:
$
Y_{i,z} =
\begin{cases}
x_{i,j,E} & \text{khi } z = 3(j - 1) + 1 \\
x_{i,j,N} & \text{khi } z = 3(j - 1) + 2
\end{cases}
$

10. Trong mỗi **7 ngày liên tiếp**, có **tối đa 4 ca tối hoặc ca đêm**
     $$\bigwedge_{i = 1}^n (\bigwedge_{j = 1}^ {d - 6} \sum_{j'= j} ^ {j + 6}  X_{i, j', E} + X_{i, j', N} \le 4 )  $$

11. **Không được phép làm ca đêm vào các ngày liên tiếp**
    $$\bigwedge_{i = 1}^n (\bigwedge_{j = 1}^ {d - 1} \sum_{j'= j} ^ {j + 1}  X_{i, j', N} \le 1 )  $$

## IV. Code

In [13]:
import math
from pysat.solvers import Glucose3
from encoding.AMKLadderEncoder import AMKLadderEncoder

## V. Mở rộng

**Đặt vấn đề**  
1. Nếu có thể làm tối đa số ca trong 1 ngày
    - Bản chất nếu không có công thức 1 thì bài toán này không thể làm theo cách trên. Vì có điều kiện 1 nên mỗi ngày sẽ chỉ có 2 trạng thái là `làm 1 ca` hoặc không làm ca nào nên bài toán đơn giản
    - Nhưng nếu không có điều kiện 1, một ngày sẽ có thể làm nhiều ca nên không thể dùng tham số $O$ như một phương pháp để giúp bài toán ngắn ngọn hơn 
2. Nếu chỉ được làm giới hạn số ca trong 1 ngày
    - Lúc này ta vẫn làm như cách có thể làm tối đa 1 này nhưng ta phải thêm các dàng buộc AMK cho từng ngày 

**Viết lại các ràng buộc có tham số $O$**

1. Trong mỗi **7 ngày liên tiếp**, mỗi nhân viên làm **tối đa 6 ca làm việc** 
   $$\bigwedge_{i = 1} ^ n \bigwedge_{j = 1} ^ {d - 6} \sum_{j' = j} ^ {j + 6} \sum_{k \in \{D, E, N \}} X_{i, j', k} \le 6$$
   

$(X_{1,1,D} + X_{1,1,E} + X_{1, 1, N}) + (X_{1,2,D} + X_{1,2,E} + X_{1, 2, N}) + (X_{1,3,D} + X_{1,3,E} + X_{1, 3, N}) + (X_{1,4,D} + X_{1,4,E} + X_{1, 4, N}) + \dots +   (X_{1,7,D} + X_{1,7,E} + X_{1,7,N})  \le 6$ 
$\ \ \\ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \  (X_{1,2,D} + X_{1,2,E} + X_{1, 2, N}) + (X_{1,3,D} + X_{1,3,E} + X_{1, 3, N}) + (X_{1,4,D} + X_{1,4,E} + X_{1, 4, N}) + \dots +   (X_{1,7,D} + X_{1,7,E} + X_{1,7,N}) + (X_{1,8,D} + X_{1,8,E} + X_{1,8,N})  \le 6$ 
$ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ (X_{1,3,D} + X_{1,3,E} + X_{1,3,N}) + (X_{1,4,D} + X_{1,4,E} + X_{1,4,N}) + \dots + (X_{1,7,D} + X_{1,7,E} + X_{1,7,N})+ (X_{1,8,D} + X_{1,8,E} + X_{1,8,N}) + (X_{1,9,D} + X_{1,9,E} + X_{1,9,N}) \le 6$

> Với những điều kiện dạng hình cầu thang nghiêng hơn "mặc định" như thế này thì ta vẫn encoding như sequence constraint như bình thường. Bằng cách nổ xung các biểu thức trung gian 

$(X_{1,1,D} + X_{1,1,E} + X_{1, 1, N}) + (X_{1,2,D} + X_{1,2,E} + X_{1, 2, N}) + (X_{1,3,D} + X_{1,3,E} + X_{1, 3, N}) + (X_{1,4,D} + X_{1,4,E} + X_{1, 4, N}) + \dots +   (X_{1,7,D} + X_{1,7,E} + X_{1,7,N})  \le 6$ 
$ \ \ \ \\ \ \\ \ \ \ \ \ \ \ \  \ \  \ \ \\ \ \underline{X_{1,1,E} + X_{1, 1, N}+ (X_{1,2,D} + X_{1,2,E} + X_{1, 2, N}) + (X_{1,3,D} + X_{1,3,E} + X_{1, 3, N}) + (X_{1,4,D} + X_{1,4,E} + X_{1, 4, N}) + \dots +   (X_{1,7,D} + X_{1,7,E} + X_{1,7,N}) + X_{1,8,D}  \le 6 }$ 
$ \ \ \ \\ \ \\ \ \ \ \ \ \ \  \ \ \ \ \ \ \  \ \ \ \ \ \ \ \ \ \ \ \ \ \ \underline{X_{1, 1, N}+ (X_{1,2,D} + X_{1,2,E} + X_{1, 2, N}) + (X_{1,3,D} + X_{1,3,E} + X_{1, 3, N}) + (X_{1,4,D} + X_{1,4,E} + X_{1, 4, N}) + \dots +   (X_{1,7,D} + X_{1,7,E} + X_{1,7,N}) + X_{1,8,D} + X_{1,8,E} \le 6} $ 
$\  \ \ \ \ \ \ \ \ \ \\ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \  (X_{1,2,D} + X_{1,2,E} + X_{1, 2, N}) + (X_{1,3,D} + X_{1,3,E} + X_{1, 3, N}) + (X_{1,4,D} + X_{1,4,E} + X_{1, 4, N}) + \dots +   (X_{1,7,D} + X_{1,7,E} + X_{1,7,N}) + (X_{1,8,D} + X_{1,8,E} + X_{1,8,N})  \le 6$ 
$$\dots$$ 
$ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ \ (X_{1,3,D} + X_{1,3,E} + X_{1,3,N}) + (X_{1,4,D} + X_{1,4,E} + X_{1,4,N}) + \dots + (X_{1,7,D} + X_{1,7,E} + X_{1,7,N})+ (X_{1,8,D} + X_{1,8,E} + X_{1,8,N}) + (X_{1,9,D} + X_{1,9,E} + X_{1,9,N}) \le 6$



$$\bigwedge_{i = 1} ^ n (\bigwedge_{j = 1} ^ {d - 6} \sum_{j' = j} ^ {j + 6} \sum_{k \in \{D, E, N \}} X_{i, j', k} \le 6) \equiv \bigwedge_{i = 1} ^ n (\bigwedge_{m = 1} ^ {3d - 20} \sum_{z = m} ^ {m + 20} Y_{i, z} \le 6)$$ 
Trong đó:
$
Y_{i,z} =
\begin{cases}
x_{i,j,D} & \text{khi } z = 3(j - 1) + 1 \\
x_{i,j,E} & \text{khi } z = 3(j - 1) + 2 \\
x_{i,j,N} & \text{khi } z = 3(j - 1) + 3
\end{cases}
$

> Nếu chỉ được làm $s$ ca một ngày


$$ \bigwedge_{i = 1} ^ n \bigwedge_{j = 1} ^ d \sum_{k \in \{D, E, N\}} X_{i,j,k} \le s$$